<a href="https://colab.research.google.com/github/Gan4x4/cv/blob/main/Detection/Detection_task.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Постановка задачи детектирования

На предыдущих занятиях мы подробно рассматривали задачу классификации изображений.

В некоторых приложениях, нам важно знать не только класс объекта, но и информацию о его **пространственном положении**. Так автопилот должен знать, **где находится объект**, чтобы его объехать. А хирургу важно знать не только расположение опухоли, но и ее **точную границу**.


<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/classification_semantic_segmentation.png" width="600"></center>

Выделяют следующие задачи:
- **Классификация** (Classification). Нужно определить класс объекта на изображении, но при этом расположение объекта не важно.
- **Детектирование** (Object Detection). Нужно определить индивидуальные **объекты** и **выделить области** (например, прямоугольники) в которых они локализованы. Например: подсчет количества китов на спутниковом снимке.
- **Сегментация** (Segmentation). Нужно определить, какие фрагменты изображения принадлежат определенным объектам/классам. При этом важно знать не только **расположение**, но и **границу объектов**.
    - **Семантическая сегментация** (Semantic segmentation) — не выделяет индивидуальные объекты, **важен только тип (класс) объекта**, которым занят конкретный пиксель. Например: определение площади леса на спутниковом снимке.
    - **Сегментация экземпляров** (Instance segmentation) — выделяет **индивидуальные объекты** и их границы. Например: выделение клеток на снимке с электронного микроскопа.
    - **Паноптическая сегментация** (Panoptic segmentation) — выделяет **индивидуальные объекты** и определяет их **класс**. Комбинирует семантическую сегментацию и сегментацию экземпляров. Например, для автопилота важно не только знать, дорога перед ним или автомобили, но и понимать, где находится ближний автомобиль, а где — дальний.

# Детектирование (Object detection)


<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/detection.png" width="400"></center>

**Детектирование** — задача компьютерного зрения, в которой требуется **определить местоположение** конкретных **объектов** на изображении. Вычислять точные границы объектов не требуется, достаточно **определить ограничивающие прямоугольники (bounding boxes)**, в которых находятся объекты.


В общем случае:
- объекты могут принадлежать к **различным классам**,
- объектов каждого класса на изображении может **не быть** или **быть несколько**.

Детекция — одна из самых бурно развивающихся областей нейронных сетей. В рамках данной лекции мы рассмотрим **некоторые идеи и архитектуры**.

## Детектирование одного объекта

Начнём с **простой ситуации**. Нас интересуют объекты только одного класса. Мы знаем, что:
- объект **точно есть** на изображении,
- объект на изображении **только один**,
- объект на изображении **принадлежит к одному классу**.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/one_object.png" width="500"></center>


**Вход модели**: изображение.

**Выход модели**: область (bounding box), в которой объект локализован, определяется набором координат вершин*.

\* *Если наложить условие, что стороны многоугольника должны быть параллельны сторонам изображения, то можно ограничиться предсказанием 2-х координат.*

### Предсказание координат

Задачу **семантической сегментации** мы свели к **попиксельной классификации**. Задачу **предсказания координат bounding box** мы попробуем свести к задаче **регрессии**, т.к. мы **предсказываем набор чисел**.



В зависимости от требований эти числа могут нести разный смысл, например:

* координаты центра + ширина и высота,
* координаты правого верхнего и левого нижнего углов,
* координаты вершин многоугольника,
* ключевые точки (определение позы, распознавание лиц и т.д.),
* ...

Но в любом случае задача остается регрессионной.

<center><img src ="https://edunet.kea.su/repo/EduNet-web_dependencies/dev-2.3/L11/predict_key_points.png" width="700"></center>

<center><em>Примеры предсказывания точек</em></center>

<center><em>Sources: <a href="https://blog.roboflow.com/intro-to-computer-vision/">Introduction to Computer Vision</a>, <a href="https://www.researchgate.net/publication/269683682_Cascade_of_Forests_for_Face_Alignment">Cascade of Forests for Face Alignment</em></center>

**Простое решение** для этой задачи:

Берем сверточную сеть и меняем последний полносвязный слой таким образом, чтобы количество выходов совпадало с количеством координат, которые нам нужно предсказать.

Так для предсказания двух точек потребуется четыре выхода: ( x1, y1, x2, y2).




In [ ]:
import torch
from torch import nn
from torchvision.models import resnet18

torch.manual_seed(42)

# Load pretrained model
resnet_detector = resnet18(weights="ResNet18_Weights.DEFAULT")

# Change "head" to predict coordinates (x1,y1,x2,y2)
resnet_detector.fc = nn.Linear(resnet_detector.fc.in_features, 4)  # x1,y1,x2,y2

Для обучения такой модели придется заменить функцию потерь на регрессионную, например, MSE.

In [ ]:
criterion = nn.MSELoss()

# This is a random example. Don't expect good results
input = torch.rand((1, 3, 224, 224))
target = torch.tensor([[0.1, 0.1, 0.5, 0.5]])  # x1,y1 x2,y2 or x,y w,h
print(f"Target: {target}")
output = resnet_detector(input)
loss = criterion(output, target)
print(f"Output: {output}")
print(f"Loss: {loss}")

Координаты обычно предсказываются в процентах от длины и ширины изображения, для чего выходы модели нужно как-то нормализовать, например, добавив последним слоем сигмоиду:

In [ ]:
resnet_detector.fc = nn.Sequential(
    nn.Linear(resnet_detector.fc.in_features, 4), nn.Sigmoid()
)

resnet_detector(input)

По такому принципу работают многие модели для поиска различных ключевых точек.
Например, на лице (facial landmarks) или теле человека.

[[arxiv] 🎓 Recent Progress in Appearance-based Action Recognition (Humphreys et al., 2020)](https://arxiv.org/abs/2011.12619)

### Multitask loss

Координаты прямоугольников мы предсказывать научились. Теперь усложним задачу:
- объект **точно есть** на изображении,
- объект на изображении **только один**,
- объект на изображении может **принадлежать к разным классам**.

К задаче **локализации (регресии)** добавляется **классификация**.

Задачу классификации мы умеем решать:

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/class_prediction.png" width="800"></center>

Остается объединить классификацию с регрессией:

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/multitask_loss_0.png" width="800"></center>

Для этого нужно одновременно предсказывать:

* вероятность принадлежности к классам,
* координаты ограничивающего прямогульника (bounding box).

Тогда выход последнего слоя будет иметь размер $N + 4$, где $N$ — количество классов (1000 для ImageNet), а $4$ числа — это координаты одного bounding box ($(x1,y1,x2,y2)$ или $(x,y,w,h)$).

**Как описать функцию потерь для такой модели?**

Можно суммировать loss для классификации и loss для регрессии.

$$ \large L_\text{total} = L_\text{cross-entropy}+L_\text{mse}$$

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/multitask_loss.png" width="800"></center>

Однако значения разных loss могут иметь разные порядки, поэтому приходится подбирать весовые коэффициенты для каждого слагаемого.
В общем случае функция потерь будет иметь вид:

$$\large L_\text{total} = \sum _iw_iL_i,$$

где $w_i$ — весовые коэффициенты каждой из функций потерь. Они являются гиперпараметрами модели и требуют подбора.

### Подбор весов для loss




Можно подбирать веса компонентов loss в процессе обучения. Для этого к модели добавляется дополнительный слой:

[[arxiv] 🎓 Multi-Task Learning Using Uncertainty to Weigh Losses for Scene Geometry and Semantics (Alex Kendall et al., 2018)](https://arxiv.org/abs/1705.07115)

[[git] 🐾 Пример реализации MultiTask learning](https://github.com/Hui-Li/multi-task-learning-example-PyTorch/blob/master/multi-task-learning-example-PyTorch.ipynb)

## Детектирование нескольких объектов

Как быть, если объектов несколько? Как мы помним:

- объекты могут принадлежать к **различным классам**,
- объектов каждого класса на изображении может **не быть** или **быть несколько**.

Для каждого объекта нужно вернуть координаты $(x, y, w, h)$ и класс $(0 .. N)$.
Соответственно, количество выходов модели надо увеличивать.

Но нам неизвестно заранее, сколько объектов будет на изображении:

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/object_detection_multiple_object.png" width="650"></center>

### Cкользящее окно

Одним из вариантов решения этой проблемы является применение классификатора ко всем возможным местоположениям объектов. Классификатор предсказывает, есть
ли на выбранном фрагменте изображения один из интересующих нас объектов. Если нет, то фрагмент классифицируется как "фон".

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/naive_way_object_detection_multiple_object.gif" width="700"></center>




Проблемой данного подхода является необходимость применять классификатор к огромному количеству различных фрагментов, что крайне дорого с точки зрения вычислений.

### Object proposal

Чтобы сократить количество возможных окон для детекции, использовали различные идеи:

1. **Использовать эвристику для нахождения потенциального положения объектов**. Области интереса (regions of interest — RoI) находятся с помощью алгоритма [Selective search 🎓[article]](https://ivi.fnwi.uva.nl/isis/publications/bibtexbrowser.php?key=UijlingsIJCV2013&bib=all.bib). Он разбивает картинку на мелкие кусочки, которые объединяет в области интереса, руководствуясь "похожестью" цвета, размера, текстуры и положения. Архитектура: Region-Based Convolutional Networks: [R-CNN 🎓[article]](https://ieeexplore.ieee.org/document/7112511).

2. **Заменить эвристический алгоритм для нахождения RoI на нейронную сеть**. Детектор состоит из 2-х ступеней: первая предполагает положение RoI (за один проход), вторая — предсказывает точные границы, есть ли там объект и к какой категории он относится (по одному проходу для каждого RoI). Архитектура: Two stage detectors:  [Faster R-CNN 🎓[arxiv]](https://arxiv.org/abs/1506.01497).

3. **Отказаться от двухступенчатой архитектуры**. Давайте заранее зададим центры и размеры окон (anchor), которые плотно покрывают все изображение/карты признаков.
Архитектуры: Single Shot Detector ([SSD 🎓[arxiv]](https://arxiv.org/abs/1512.02325)), You Only Look Once v2 ([YOLOv2 🎓[arxiv]](https://arxiv.org/pdf/1612.08242)).

4. **Отказаться от anchor**. Нейросеть предсказывает вероятность нахождения объекта и расстояния для границ его bounding box для каждой точки на карте признаков. Архитектура: Anchor free [FCOS 🎓[arxiv]](https://arxiv.org/abs/1904.01355).

5. **Взять ViT, делать предсказание для каждого patch**. Архитектура: [DETR 🎓[arxiv]](https://arxiv.org/abs/2005.12872).

<img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/detector_types.png" width="1000">

**Общая схема работы детектора**:

* R-CNN: изображение $\xrightarrow{\text{Эвристика}}$ RoI $\xrightarrow{\text{CNN}}$ уточненный bounding box и его категория.
* Faster R-CNN: изображение $\xrightarrow{\text{Region proposal network}}$ RoI в пространстве признаков $\xrightarrow{\text{CNN}}$ уточненный bounding box и его категория.
* One stage: данные $\xrightarrow{\text{Статистика}}$ anchors,

    изображение + anchors $\xrightarrow{\text{CNN}}$ уточненные bounding boxes и их категории.
* Anchor free: изображение $\xrightarrow{\text{CNN}}$ bounding boxes и их категории.
* Transformer: изображение, разбитое на patch $\xrightarrow{\text{Transformer}}$ bounding boxes и их категории.

<img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/detector_scheme.png" width="1000">

Loss складывается из loss для классификации $L_\text{conf}$ и loss для детекции $L_\text{loc}$:

$$\large L_\text{det} = L_\text{conf} + \alpha L_\text{loc}$$

При этом в $L_\text{loc}$ учитываются **не все** предсказанные bounding box, а только те, которые наилучшим образом пересекаются с GT (bbox из разметки). Фильтрация может проходить по порогу или при помощи алгоритма.

### Backbone, neck and head

Первые **детекторы** были похожи на **сети для классификации**.  Изображение проходило через последовательность сверточных слоев, которые извлекали признаки (backbone), на этих признаках делались предсказания (head).

<center><img src ="https://edunet.kea.su/repo/EduNet-web_dependencies/dev-2.3/L11/yolov1_architecture.png" width="1000"></center>

<center> Архитектура YOLOv1

Source: <a href ="https://arxiv.org/pdf/1506.02640">You Only Look Once: Unified, Real-Time Object Detection</a></center>



У такого подхода есть проблема. Он **хорош для обнаружения крупных объектов**, но плохо различает мелкие объекты на изображении. Эта проблема решена в современных детекторах. Для этого:
- используют **несколько "голов"** (head) детекции, которые работают с признаками разных размеров,
- используют **"шею"** (neck), в которой признаки разных размеров объединяются и перемешиваются.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/backbone_neck_head.png" width="900"></center>

<center><em>Backbone, neck и head в архитектуре YOLOv11 </em></center>

Итого:
- **Backbone** (хребет) — привычная нам пирамида для извлечения признаков (Feature Pyramid Network (FPN)),
- **Neck** (шея) — блок, который собирает признаки с разных уровней и перемешивает их,
- **Head** (голова) — принимает решение о наличии, положении и категории объекта.

### One-to-many vs one-to-one

Большинство детекторов учится в режиме one-to-many. К одному объекту с ограничивающим прямоугольником (bounding box) "притягиваются" сразу несколько объектов на выходе модели. Это дает [более устойчивое обучение 🎓[arxiv]](https://arxiv.org/pdf/2405.14458), т.к. ошибка усредняется по нескольким прямоугольникам.

Предсказание такого детектора требует постобработки.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/non_max_suppression.png" width="650"></center>

#### NMS

Есть несколько подходов к удалению лишних прямоугольников.
Наиболее известный — NMS (non-maximum suppression). Его задача — избавиться от bounding boxes, которые накладываются на истинный. Последовательность действий:

1. Из списка возможных bounding boxes выбирается прогноз с наибольшей уверенностью, что в нем расположен объект (confidence score). Этот bounding box добавляется в итоговый список и удаляется из исходного списка.
2. Для выбранного bounding box считается $\text{IoU}$ с каждым из списка возможных bounding boxes. Если $\text{IoU}$ больше порогового $\text{IoU}_{\text{threshold}}$ — bounding box удаляется из списка возможных (считается, что это bounding box для того же объекта, который добавлен в итоговый список).
3. Шаги 1–2 повторяются, пока список возможных bounding boxes не опустеет.

Значение $\text{IoU}_{\text{threshold}}$ является гиперпараметром.

При низком значении $\text{IoU}_{\text{threshold}}$ частично пересекающиеся объекты могут быть удалены. При высоком значениии $\text{IoU}_{\text{threshold}}$ для одного объекта может остаться несколько масок.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L11/out/non_max_suppression_pseudo_code.png" width="600"></center>
<center><em>Пример картинки, на которой NMS может работать некорректно</em></center>

В PyTorch алгоритм NMS доступен в модуле `torchvision.ops.nms` [🛠️[doc]](https://pytorch.org/vision/stable/generated/torchvision.ops.nms.html)
:

`torchvision.ops.nms(boxes, scores, iou_threshold)`,
где:
* `boxes` — массив bounding boxes,
* `scores` — предсказанные оценки,
* `iou_threshold` — порог IoU, NMS отбрасывает все перекрывающиеся поля с IoU > `iou_threshold`.

Существуют модификации алгоритма:
- [[arxiv] 🎓 Soft NMS](https://arxiv.org/pdf/1704.04503)
- [[arxiv] 🎓 Learnable NMS](https://arxiv.org/pdf/1705.02950)



#### NMS-free

NMS — достаточно тяжелая и не всегда точная операция, от которой хотелось бы избавиться на inference (при применении модели). Существуют one-to-one подходы:
- В [DETR 🎓[arxiv]](https://arxiv.org/pdf/2005.12872) для избавления от NMS используют encoder-decoder архитектуру трансформера, генерировать последовательность объектов относительно небольшой длины и loss, основанный на [венгерском алгоритме 📚[wiki]](https://ru.wikipedia.org/wiki/%D0%92%D0%B5%D0%BD%D0%B3%D0%B5%D1%80%D1%81%D0%BA%D0%B8%D0%B9_%D0%B0%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC), для сопоставления объектов на входе и на выходе.
- В [YOLOv10 🎓[arxiv]](https://arxiv.org/pdf/2405.14458) появилась концепция Dual Label Assignment: при обучении модели одновременно обучаются две головы: one-to-many, которая обеспечивает стабильное обучение, и one-to-one, которая используется на inference. В loss добавлено дополнительное ограничение, чтобы предсказания двух голов не расходились.

<center><img src ="https://edunet.kea.su/repo/EduNet-web_dependencies/dev-2.3/L11/yolov10_dual_label.jpg" width="768"></center>
<center> Source: <a href ="https://arxiv.org/pdf/2405.14458">YOLOv10: Real-Time End-to-End Object Detection</a></center>